# Тема 4. Деревья решений и ансамблевые методы

**Лабораторная работа на UCI Adult (Census Income)**

Цель — построить воспроизводимый pipeline для смешанных табличных данных, сравнить дерево решений, Random Forest, Extra Trees и градиентный бустинг, исследовать переобучение и pruning, интерпретировать модель и провести ограниченный аудит справедливости.

**Протокол эксперимента.** `random_state=42`, стратифицированное разделение 80/20 и 5-fold `StratifiedKFold`. Test-набор не применяется для выбора гиперпараметров или порога: порог выбирается по out-of-fold прогнозам train. Финальная оценка на test выполняется после выбора конфигураций. Все preprocessing-операции входят в `Pipeline`.

## 0. Среда и воспроизводимость

При необходимости раскомментируйте команду установки и выполните ячейку один раз. Для загрузки Adult через UCI требуется доступ к интернету.

In [1]:
# %pip install -q ucimlrepo pandas numpy matplotlib seaborn scikit-learn jupyter
import sys, platform, time, json, os, pickle, warnings
from time import perf_counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from IPython.display import display, Markdown
from sklearn.base import clone

RANDOM_STATE = 42
# Ограничиваем параллелизм, чтобы CV и модели не запускали вложенные пулы потоков.
N_JOBS = min(2, os.cpu_count() or 1)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("Parallel jobs:", N_JOBS)

Python: 3.14.6
Platform: Linux-7.1.6-1-cachyos-x86_64-with-glibc2.44
pandas: 3.0.6
numpy: 2.5.4
scikit-learn: 1.9.1
Parallel jobs: 2


## 1. Загрузка данных

Источник: UCI Adult, DOI `10.24432/C5XW20`. В исходных полях `?` может обозначать пропущенное значение, поэтому этот маркер нормализуется в `NaN` до описательной статистики и разбиения. Признаки `sex` и `race` сохраняются: далее они нужны для аудита, а не для причинных выводов.

In [2]:
from ucimlrepo import fetch_ucirepo

adult = fetch_ucirepo(id=2)
X = adult.data.features.copy()
y_raw = adult.data.targets.iloc[:, 0].astype(str)

# Убираем случайные пробелы в строковых значениях и приводим маркер '?' к пропуску.
for col in X.select_dtypes(include=["object", "string"]).columns:
    X[col] = X[col].map(lambda value: value.strip() if isinstance(value, str) else value)
X = X.replace(r"^\s*\?\s*$", np.nan, regex=True)

y_raw = y_raw.str.replace(".", "", regex=False).str.strip()
y = y_raw.eq(">50K").astype(int).rename("income_gt_50k")

assert len(X) == len(y), "Число строк в X и y не совпадает"
assert set(y.unique()).issubset({0, 1}), "Цель должна быть бинарной"
print(f"Размер X: {X.shape}; число целевых меток: {len(y)}")
print(f"Доля класса >50K: {y.mean():.4f}")
display(X.head())

ModuleNotFoundError: No module named 'ucimlrepo'

### 1.1. Паспорт данных и качество

Таблица ниже содержит типы, число пропусков и количество уникальных значений. Дубликаты признаков и полных записей считаются отдельно: одинаковые признаки ещё не означают, что целевые метки совпадают.

**Как читать чувствительные признаки.** Различия модели между категориями `sex` и `race` являются описательными ассоциациями в этом наборе данных. Они не доказывают причинное влияние пола или расы на доход. Другие признаки, например профессия, образование или семейный статус, могут выступать прокси социально чувствительных характеристик.

Потенциальные риски качества данных: (1) пропуски, обозначенные `?`; (2) исторический характер переписных данных и возможная нерепрезентативность для современных условий; (3) дубликаты/зависимые наблюдения и неоднозначная трактовка целевого порога дохода.

In [ ]:
summary = pd.DataFrame({
    "dtype": X.dtypes.astype(str),
    "missing_n": X.isna().sum(),
    "missing_pct": (100 * X.isna().mean()).round(2),
    "n_unique": X.nunique(dropna=False),
}).sort_values(["missing_n", "n_unique"], ascending=False)
display(summary)

full_data = X.copy()
full_data["__target__"] = y
print("Дубликаты только по признакам:", int(X.duplicated().sum()))
print("Полные дубликаты (признаки + цель):", int(full_data.duplicated().sum()))

target_table = y.value_counts().sort_index().rename(index={0: "<=50K", 1: ">50K"}).to_frame("n")
target_table["share_pct"] = (100 * target_table["n"] / target_table["n"].sum()).round(2)
display(target_table)

ax = target_table["n"].plot(kind="bar", figsize=(6, 4), rot=0)
ax.set_title("Распределение целевого класса Adult")
ax.set_xlabel("Годовой доход")
ax.set_ylabel("Число наблюдений")
plt.tight_layout()
plt.show()

## 2. Train/test и preprocessing

Фиксируем `test_size=0.20`, `stratify=y`, `random_state=42`. Стратификация сохраняет близкую долю положительного класса в train и test. Случайное разбиение предполагает, что наблюдения достаточно независимы; оно не моделирует временной сдвиг и может переоценить качество на новых популяциях.

Все статистики imputation и словарь категорий обучаются только на обучающей части соответствующего фолда. Масштабирование числовых признаков деревьям не требуется.

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols),
], remainder="drop")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("Train:", X_train.shape, "Test:", X_test.shape)
print(f"Доля >50K в train: {y_train.mean():.4f}")
print("Числовых признаков:", len(num_cols), "Категориальных:", len(cat_cols))

### 2.1. Контроль утечки данных

Нельзя до split вычислять медианы/моды для imputation, обучать `OneHotEncoder`, отбирать признаки по всей выборке, масштабировать данные по общим статистикам или подбирать гиперпараметры по test. Даже если преобразование не использует `y`, его обучение на test может раскрыть распределение будущих данных. `handle_unknown="ignore"` позволяет обработать категорию, появившуюся в validation/test, но отсутствовавшую при fit: для неё создаётся нулевой OHE-вектор вместо ошибки. Неизвестная категория при этом всё равно несёт неопределённость.

## 3. Baselines: Dummy и логистическая регрессия

Используются одинаковые стратифицированные фолды и метрики: ROC AUC, PR AUC (Average Precision), balanced accuracy и log loss. Для log loss знак `neg_log_loss`, который возвращает scikit-learn, меняется на обычный положительный log loss. Выводятся среднее и стандартное отклонение по 5 фолдам.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score, f1_score, log_loss
from sklearn.preprocessing import StandardScaler

scoring = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "bal_acc": "balanced_accuracy",
    "neg_log_loss": "neg_log_loss",
}
logreg_prep = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_cols),
], remainder="drop")

baseline_estimators = {
    "Dummy": DummyClassifier(strategy="prior"),
    "Logistic Regression": LogisticRegression(max_iter=1500, solver="lbfgs"),
}
baseline_pipelines = {}
baseline_cv_stats = {}
baseline_rows = []
for name, estimator in baseline_estimators.items():
    current_prep = logreg_prep if name == "Logistic Regression" else prep
    pipe = Pipeline([("prep", clone(current_prep)), ("model", clone(estimator))])
    out = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring,
                         n_jobs=N_JOBS, return_train_score=True, error_score="raise")
    baseline_pipelines[name] = pipe
    baseline_cv_stats[name] = {
        "roc_auc_mean": float(out["test_roc_auc"].mean()),
        "roc_auc_std": float(out["test_roc_auc"].std(ddof=1)),
        "pr_auc_mean": float(out["test_pr_auc"].mean()),
        "bal_acc_mean": float(out["test_bal_acc"].mean()),
        "log_loss_mean": float((-out["test_neg_log_loss"]).mean()),
        "fit_s_mean": float(out["fit_time"].mean()),
    }
    row = {"model": name}
    for metric in ["roc_auc", "pr_auc", "bal_acc"]:
        row[f"{metric}_mean"] = out[f"test_{metric}"].mean()
        row[f"{metric}_std"] = out[f"test_{metric}"].std(ddof=1)
    row["log_loss_mean"] = (-out["test_neg_log_loss"]).mean()
    row["log_loss_std"] = out["test_neg_log_loss"].std(ddof=1)
    row["fit_s_mean"] = out["fit_time"].mean()
    baseline_rows.append(row)

baseline_cv_table = pd.DataFrame(baseline_rows).set_index("model")
display(baseline_cv_table.round(4))

## 4. Короткое диагностическое дерево и правила

Используем интерпретируемое дерево `max_depth=4`, `min_samples_leaf=50`. Оно показывает простые сегменты наблюдений, но не должно рассматриваться как причинная модель. Текстовые правила и визуализация относятся только к train.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

tree_pipe = Pipeline([
    ("prep", clone(prep)),
    ("model", DecisionTreeClassifier(max_depth=4, min_samples_leaf=50,
                                      random_state=RANDOM_STATE)),
])
tree_pipe.fit(X_train, y_train)
tree_model = tree_pipe.named_steps["model"]
feature_names = tree_pipe.named_steps["prep"].get_feature_names_out()
train_tree_p = tree_pipe.predict_proba(X_train)[:, 1]
tree_cv_out = cross_validate(tree_pipe, X_train, y_train, cv=cv, scoring="roc_auc",
                             return_train_score=True, n_jobs=N_JOBS, error_score="raise")
print(export_text(tree_model, feature_names=list(feature_names), max_depth=4))
print(f"Train ROC AUC: {roc_auc_score(y_train, train_tree_p):.4f}")
print(f"5-fold CV ROC AUC: {tree_cv_out['test_score'].mean():.4f} ± {tree_cv_out['test_score'].std(ddof=1):.4f}")
print(f"Число узлов: {tree_model.tree_.node_count}; листьев: {tree_model.get_n_leaves()}; глубина: {tree_model.get_depth()}")

plt.figure(figsize=(22, 10))
plot_tree(tree_model, feature_names=feature_names,
          class_names=["<=50K", ">50K"], filled=True, max_depth=3, fontsize=7)
plt.title("Первые уровни диагностического дерева (обучающая выборка)")
plt.tight_layout()
plt.show()

### 4.1. Три пути к листьям

Ниже автоматически выбираются листья с разными оценёнными долями класса `>50K`, при достаточном числе обучающих наблюдений. Условия читаются как логические правила. Они описывают подгруппы в данном наборе, но не означают, что изменение одного признака вызовет изменение дохода. Разумно проверить устойчивость правил при другом split и к чувствительным прокси-признакам.

In [ ]:
# Число наблюдений и фактическая доля положительного класса для каждого листа.
X_train_tree = tree_pipe.named_steps["prep"].transform(X_train)
leaf_ids = tree_model.apply(X_train_tree)
leaf_probabilities = tree_model.predict_proba(X_train_tree)[:, 1]
leaf_rows = []
for leaf in np.unique(leaf_ids):
    mask = (leaf_ids == leaf)
    leaf_rows.append({
        "leaf_id": int(leaf),
        "n_train": int(mask.sum()),
        "positive_rate": float(y_train.to_numpy()[mask].mean()),
        "tree_probability_class1": float(leaf_probabilities[mask].mean()),
    })
leaf_table = pd.DataFrame(leaf_rows)
eligible_leaves = leaf_table[leaf_table["n_train"] >= 100].copy()
if len(eligible_leaves) >= 3:
    chosen_leaf_ids = [
        int(eligible_leaves.loc[eligible_leaves["positive_rate"].idxmin(), "leaf_id"]),
        int(eligible_leaves.iloc[(eligible_leaves["positive_rate"] - 0.5).abs().argmin()]["leaf_id"]),
        int(eligible_leaves.loc[eligible_leaves["positive_rate"].idxmax(), "leaf_id"]),
    ]
else:
    chosen_leaf_ids = leaf_table.sort_values("n_train", ascending=False)["leaf_id"].head(3).astype(int).tolist()
chosen_leaf_ids = list(dict.fromkeys(chosen_leaf_ids))

t = tree_model.tree_
parent = {}
for node in range(t.node_count):
    if t.children_left[node] != -1:
        parent[int(t.children_left[node])] = (node, "left")
        parent[int(t.children_right[node])] = (node, "right")

def path_to_leaf(leaf_id):
    conditions = []
    node = int(leaf_id)
    while node in parent:
        par, side = parent[node]
        feat = feature_names[t.feature[par]]
        thr = t.threshold[par]
        if side == "left":
            conditions.append(f"{feat} <= {thr:.3g}")
        else:
            conditions.append(f"{feat} > {thr:.3g}")
        node = par
    return " AND ".join(reversed(conditions)) if conditions else "(корень дерева)"

rule_rows = []
for leaf_id in chosen_leaf_ids:
    info = leaf_table.loc[leaf_table["leaf_id"] == leaf_id].iloc[0]
    rule_rows.append({
        "leaf_id": leaf_id,
        "n_train": int(info["n_train"]),
        "positive_rate_train": round(float(info["positive_rate"]), 4),
        "tree_probability_class1": round(float(info["tree_probability_class1"]), 4),
        "rule": path_to_leaf(leaf_id),
    })
rule_table = pd.DataFrame(rule_rows)
display(rule_table)

## 5. Кривая сложности: глубина × размер листа

Исследуется полный набор комбинаций `max_depth=1…20` и `min_samples_leaf ∈ {1, 5, 20, 50, 200}`. Сравниваются train и CV ROC AUC. Для выбора простой конфигурации используется правило одной стандартной ошибки: среди вариантов в пределах одной стандартной ошибки от лучшего CV-результата выбирается более простое дерево (меньшая глубина, больший лист).

In [ ]:
complexity_rows = []
leaf_values = [1, 5, 20, 50, 200]
for leaf_size in leaf_values:
    for depth in range(1, 21):
        pipe = Pipeline([
            ("prep", clone(prep)),
            ("model", DecisionTreeClassifier(max_depth=depth,
                                              min_samples_leaf=leaf_size,
                                              random_state=RANDOM_STATE)),
        ])
        out = cross_validate(pipe, X_train, y_train, cv=cv, scoring="roc_auc",
                             return_train_score=True, n_jobs=N_JOBS, error_score="raise")
        complexity_rows.append({
            "max_depth": depth,
            "min_samples_leaf": leaf_size,
            "train_auc": float(out["train_score"].mean()),
            "cv_auc": float(out["test_score"].mean()),
            "cv_std": float(out["test_score"].std(ddof=1)),
            "fit_s_mean": float(out["fit_time"].mean()),
        })

complexity_df = pd.DataFrame(complexity_rows)
curve_leaf20 = complexity_df[complexity_df["min_samples_leaf"] == 20].sort_values("max_depth")
plt.figure(figsize=(10, 5))
plt.plot(curve_leaf20["max_depth"], curve_leaf20["train_auc"], marker="o", label="Train ROC AUC")
plt.plot(curve_leaf20["max_depth"], curve_leaf20["cv_auc"], marker="o", label="5-fold CV ROC AUC")
plt.fill_between(curve_leaf20["max_depth"],
                 curve_leaf20["cv_auc"] - curve_leaf20["cv_std"],
                 curve_leaf20["cv_auc"] + curve_leaf20["cv_std"], alpha=0.18,
                 label="CV ± 1 SD")
plt.xlabel("max_depth")
plt.ylabel("ROC AUC")
plt.title("Кривая сложности дерева (min_samples_leaf=20)")
plt.legend()
plt.tight_layout()
plt.show()

heat = complexity_df.pivot(index="min_samples_leaf", columns="max_depth", values="cv_auc")
plt.figure(figsize=(15, 4.5))
sns.heatmap(heat, cmap="viridis", annot=False, cbar_kws={"label": "Mean 5-fold CV ROC AUC"})
plt.xlabel("max_depth")
plt.ylabel("min_samples_leaf")
plt.title("CV ROC AUC для сетки параметров дерева")
plt.tight_layout()
plt.show()

best_row = complexity_df.loc[complexity_df["cv_auc"].idxmax()]
one_se_cutoff = float(best_row["cv_auc"] - best_row["cv_std"] / np.sqrt(cv.get_n_splits()))
eligible = complexity_df[complexity_df["cv_auc"] >= one_se_cutoff].copy()
selected_row = eligible.sort_values(["max_depth", "min_samples_leaf"], ascending=[True, False]).iloc[0]
selected_tree_params = {
    "max_depth": int(selected_row["max_depth"]),
    "min_samples_leaf": int(selected_row["min_samples_leaf"]),
}
selected_tree_cv_auc = float(selected_row["cv_auc"])
selected_tree_train_pipe = Pipeline([
    ("prep", clone(prep)),
    ("model", DecisionTreeClassifier(**selected_tree_params, random_state=RANDOM_STATE)),
]).fit(X_train, y_train)
selected_tree_train_auc = float(roc_auc_score(y_train, selected_tree_train_pipe.predict_proba(X_train)[:, 1]))
print("Лучший по среднему CV ROC AUC:", best_row[["max_depth", "min_samples_leaf", "cv_auc", "cv_std"]].to_dict())
print("Порог правила 1 SD:", round(one_se_cutoff, 4))
print("Выбранная более простая конфигурация:", selected_tree_params)
print(f"Train ROC AUC выбранного дерева: {selected_tree_train_auc:.4f}; CV ROC AUC: {selected_tree_cv_auc:.4f}")

### 5.1. Интерпретация bias–variance trade-off

- **Underfitting:** малые `max_depth` и/или большие `min_samples_leaf`; модель слишком проста, train и CV ROC AUC обычно невысоки.
- **Разумный компромисс:** выбранная выше конфигурация по правилу одной стандартной ошибки — небольшая потеря среднего CV-качества относительно максимума в обмен на простоту.
- **Начало overfitting:** участок, где train ROC AUC продолжает расти, а CV ROC AUC перестаёт расти или снижается; увеличивается разрыв train–CV. Граница определяется по построенным графикам, а не задаётся заранее.

Стандартное отклонение между фолдами отражает изменчивость оценки; оно не устраняет неопределённость, связанную с возможным временным или популяционным сдвигом.

## 6. Cost-complexity pruning

Подбор `ccp_alpha`, глубины и минимального размера листа выполняется через 5-fold CV и `GridSearchCV` только на train. Ниже сравниваются выбранное дерево и непрунированный вариант по CV-ROC AUC, глубине и числу листьев. Test-метрики для выбранных моделей появятся только в общей финальной таблице.

In [ ]:
from sklearn.model_selection import GridSearchCV

prune_pipe = Pipeline([
    ("prep", clone(prep)),
    ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
])
prune_grid = {
    "model__max_depth": [5, 8, None],
    "model__min_samples_leaf": [20, 50],
    "model__ccp_alpha": [0.0, 1e-5, 1e-4, 3e-4, 1e-3],
}
prune_search = GridSearchCV(prune_pipe, prune_grid, scoring="roc_auc", cv=cv,
                            n_jobs=N_JOBS, return_train_score=True, refit=True,
                            error_score="raise")
prune_search.fit(X_train, y_train)
best_pruned_pipe = prune_search.best_estimator_
best_pruned_tree = best_pruned_pipe.named_steps["model"]
best_pruned_cv_auc = float(prune_search.best_score_)
best_pruned_params = dict(prune_search.best_params_)

# Непрунированный референс из той же сетки: max_depth=None, leaf=20, alpha=0.
raw_params = prune_search.cv_results_["params"]
raw_idx = next(i for i, p in enumerate(raw_params)
               if p["model__max_depth"] is None and p["model__min_samples_leaf"] == 20
               and p["model__ccp_alpha"] == 0.0)
raw_cv_auc = float(prune_search.cv_results_["mean_test_score"][raw_idx])

unpruned_reference = Pipeline([
    ("prep", clone(prep)),
    ("model", DecisionTreeClassifier(max_depth=None, min_samples_leaf=20,
                                      ccp_alpha=0.0, random_state=RANDOM_STATE)),
]).fit(X_train, y_train)
unpruned_tree = unpruned_reference.named_steps["model"]
pruning_comparison = pd.DataFrame([
    {"variant": "Unpruned reference", "cv_roc_auc": raw_cv_auc,
     "ccp_alpha": 0.0, "depth": unpruned_tree.get_depth(),
     "leaves": unpruned_tree.get_n_leaves()},
    {"variant": "Selected pruned tree", "cv_roc_auc": best_pruned_cv_auc,
     "ccp_alpha": float(best_pruned_params["model__ccp_alpha"]),
     "depth": best_pruned_tree.get_depth(), "leaves": best_pruned_tree.get_n_leaves()},
])
display(pruning_comparison)
print("Лучшие параметры:", best_pruned_params)
print(f"Лучший средний 5-fold CV ROC AUC: {best_pruned_cv_auc:.4f}")
print("Топ-10 конфигураций по CV ROC AUC:")
prune_results = pd.DataFrame(prune_search.cv_results_).sort_values("mean_test_score", ascending=False)
display(prune_results[["param_model__max_depth", "param_model__min_samples_leaf",
                       "param_model__ccp_alpha", "mean_train_score", "mean_test_score",
                       "std_test_score", "mean_fit_time"]].head(10).round(4))

**Как оценивать pruning.** Положительный результат — уменьшение глубины/числа листьев при сопоставимом CV ROC AUC. Если качество снижается заметно, упрощение может быть слишком сильным; если качество сохраняется, более компактное дерево предпочтительно по прозрачности. Из-за выбора по конечной сетке найденный `ccp_alpha` — лучший среди проверенных значений, а не обязательно глобальный оптимум.

## 7. Random Forest и OOB

Сначала исследуется стабилизация OOB accuracy при росте числа деревьев. Затем меняются `max_features`, `min_samples_leaf` и `max_samples`; конфигурация выбирается только по train/OOB. После этого выбранная конфигурация оценивается на тех же пяти CV-фолдах, что и другие модели. Для контроля вычислительных затрат число деревьев в финальной CV-конфигурации ограничено 400.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Для OOB-эксперимента кодируем только X_train; test здесь не используется.
oob_prep = clone(prep).fit(X_train, y_train)
X_train_ohe = oob_prep.transform(X_train)
y_train_np = y_train.to_numpy()

n_tree_grid = [25, 50, 100, 200, 400, 800]
oob_curve_model = RandomForestClassifier(
    n_estimators=n_tree_grid[0], max_features="sqrt", min_samples_leaf=5,
    bootstrap=True, oob_score=True, warm_start=True, n_jobs=1,
    random_state=RANDOM_STATE,
)
oob_curve_rows = []
curve_start = perf_counter()
for n_trees in n_tree_grid:
    oob_curve_model.set_params(n_estimators=n_trees)
    t0 = perf_counter()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        oob_curve_model.fit(X_train_ohe, y_train_np)
    oob_curve_rows.append({
        "n_estimators": n_trees,
        "oob_accuracy": float(oob_curve_model.oob_score_),
        "incremental_fit_s": perf_counter() - t0,
        "cumulative_s": perf_counter() - curve_start,
    })
oob_curve = pd.DataFrame(oob_curve_rows)
display(oob_curve.round(4))

plt.figure(figsize=(8, 4.5))
plt.plot(oob_curve["n_estimators"], oob_curve["oob_accuracy"], marker="o")
plt.xlabel("Число деревьев (n_estimators)")
plt.ylabel("OOB accuracy")
plt.title("Стабилизация OOB-оценки Random Forest")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# Выбираем минимальное число деревьев, для которого последующие OOB-оценки
# укладываются в диапазон 0.003; иначе используем верхнюю точку графика.
rf_stable_n_estimators = n_tree_grid[-1]
for n_trees in n_tree_grid:
    later = oob_curve.loc[oob_curve["n_estimators"] >= n_trees, "oob_accuracy"]
    if len(later) and (later.max() - later.min()) <= 0.003:
        rf_stable_n_estimators = int(n_trees)
        break
rf_n_estimators_selected = int(min(rf_stable_n_estimators, 400))

# Небольшой OOB-эксперимент: меняем по одному ключевому параметру.
rf_oob_candidates = [
    {"setting": "baseline", "max_features": "sqrt", "min_samples_leaf": 5, "max_samples": None},
    {"setting": "more_features", "max_features": 0.5, "min_samples_leaf": 5, "max_samples": None},
    {"setting": "larger_leaf", "max_features": "sqrt", "min_samples_leaf": 10, "max_samples": None},
    {"setting": "subsample_per_tree", "max_features": "sqrt", "min_samples_leaf": 5, "max_samples": 0.8},
]
rf_oob_rows = []
for cfg in rf_oob_candidates:
    estimator = RandomForestClassifier(
        n_estimators=100, max_features=cfg["max_features"],
        min_samples_leaf=cfg["min_samples_leaf"], max_samples=cfg["max_samples"],
        bootstrap=True, oob_score=True, n_jobs=1, random_state=RANDOM_STATE,
    )
    t0 = perf_counter()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        estimator.fit(X_train_ohe, y_train_np)
    rf_oob_rows.append({**cfg, "oob_accuracy": float(estimator.oob_score_),
                        "fit_s": perf_counter() - t0})
rf_oob_table = pd.DataFrame(rf_oob_rows).sort_values("oob_accuracy", ascending=False)
display(rf_oob_table.round(4))

best_rf_setting = rf_oob_table.iloc[0].to_dict()
selected_max_samples = best_rf_setting["max_samples"]
if pd.isna(selected_max_samples):
    selected_max_samples = None
rf_selected_params = {
    "max_features": best_rf_setting["max_features"],
    "min_samples_leaf": int(best_rf_setting["min_samples_leaf"]),
    "max_samples": selected_max_samples,
}
print(f"OOB-выбор параметров: {rf_selected_params}")
print(f"Стабилизация предложила n_estimators={rf_stable_n_estimators}; для CV выбран бюджетный вариант {rf_n_estimators_selected} деревьев.")

rf_pipe = Pipeline([
    ("prep", clone(prep)),
    ("model", RandomForestClassifier(
        n_estimators=rf_n_estimators_selected,
        max_features=rf_selected_params["max_features"],
        min_samples_leaf=rf_selected_params["min_samples_leaf"],
        max_samples=rf_selected_params["max_samples"],
        bootstrap=True, oob_score=False, n_jobs=1, random_state=RANDOM_STATE,
    )),
])
rf_cv_out = cross_validate(rf_pipe, X_train, y_train, cv=cv,
                           scoring={"roc_auc": "roc_auc", "pr_auc": "average_precision",
                                    "bal_acc": "balanced_accuracy", "accuracy": "accuracy",
                                    "neg_log_loss": "neg_log_loss"},
                           n_jobs=N_JOBS, return_train_score=True, error_score="raise")
rf_cv_auc = float(rf_cv_out["test_roc_auc"].mean())
rf_cv_auc_std = float(rf_cv_out["test_roc_auc"].std(ddof=1))
rf_cv_accuracy = float(rf_cv_out["test_accuracy"].mean())
rf_cv_fit_s = float(rf_cv_out["fit_time"].mean())
rf_cv_table = pd.DataFrame([{
    "metric": "ROC AUC", "mean_cv": rf_cv_auc, "std_cv": rf_cv_auc_std,
}, {
    "metric": "PR AUC", "mean_cv": rf_cv_out["test_pr_auc"].mean(),
    "std_cv": rf_cv_out["test_pr_auc"].std(ddof=1),
}, {
    "metric": "Balanced accuracy", "mean_cv": rf_cv_out["test_bal_acc"].mean(),
    "std_cv": rf_cv_out["test_bal_acc"].std(ddof=1),
}, {
    "metric": "Accuracy (для сравнения с OOB)", "mean_cv": rf_cv_accuracy,
    "std_cv": rf_cv_out["test_accuracy"].std(ddof=1),
}, {
    "metric": "Log loss", "mean_cv": (-rf_cv_out["test_neg_log_loss"]).mean(),
    "std_cv": rf_cv_out["test_neg_log_loss"].std(ddof=1),
}])
display(rf_cv_table.round(4))
print(f"OOB accuracy в лучшем 100-tree варианте: {best_rf_setting['oob_accuracy']:.4f}")
print(f"5-fold CV accuracy: {rf_cv_accuracy:.4f} ± {rf_cv_out['test_accuracy'].std(ddof=1):.4f}")

### 7.1. Сравнение OOB и CV

OOB accuracy и средняя 5-fold CV accuracy измеряют одну метрику разными способами, поэтому сопоставимы по величине, но не идентичны по протоколу. OOB использует bootstrap-наблюдения, не попавшие в конкретное дерево; CV использует отдельный фолд. Расхождение может возникнуть из-за вариативности разбиения, числа деревьев и того, что OOB-сравнение гиперпараметров само по себе является процедурой выбора.

## 8. Extra Trees

Extra Trees строится с тем же числом деревьев и близкими ограничениями листа, что и Random Forest. Дополнительная случайность в выборе порогов обычно уменьшает корреляцию между деревьями, но может увеличить смещение; результат зависит от данных и должен оцениваться по общим CV-фолдам.

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

extra_pipe = Pipeline([
    ("prep", clone(prep)),
    ("model", ExtraTreesClassifier(
        n_estimators=rf_n_estimators_selected,
        min_samples_leaf=rf_selected_params["min_samples_leaf"],
        max_features=rf_selected_params["max_features"],
        n_jobs=1, random_state=RANDOM_STATE,
    )),
])
extra_cv_out = cross_validate(extra_pipe, X_train, y_train, cv=cv,
                              scoring={"roc_auc": "roc_auc", "pr_auc": "average_precision",
                                       "bal_acc": "balanced_accuracy", "neg_log_loss": "neg_log_loss"},
                              n_jobs=N_JOBS, return_train_score=True, error_score="raise")
extra_cv_auc = float(extra_cv_out["test_roc_auc"].mean())
extra_cv_auc_std = float(extra_cv_out["test_roc_auc"].std(ddof=1))
extra_cv_fit_s = float(extra_cv_out["fit_time"].mean())
extra_cv_table = pd.DataFrame([{
    "model": "Extra Trees", "roc_auc_mean": extra_cv_auc, "roc_auc_std": extra_cv_auc_std,
    "pr_auc_mean": extra_cv_out["test_pr_auc"].mean(),
    "balanced_acc_mean": extra_cv_out["test_bal_acc"].mean(),
    "log_loss_mean": (-extra_cv_out["test_neg_log_loss"]).mean(),
    "fit_s_mean": extra_cv_fit_s,
}])
display(extra_cv_table.round(4))

## 9. Gradient Boosting: HistGradientBoostingClassifier

One-hot матрица для HGB создаётся плотной, поэтому она может потребовать больше памяти, чем разреженное представление для лесов. Подбираются `learning_rate`, `max_iter`, `max_leaf_nodes`, `min_samples_leaf` и `l2_regularization` через `RandomizedSearchCV`; используется early stopping. Настройка проводится только внутри train. Затем выбранная конфигурация оценивается на тех же пяти фолдах для сопоставления с RF и Extra Trees.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV, cross_validate

# В новых категориях применяется игнорирование неизвестного уровня; плотное кодирование — для HGB.
dense_prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), cat_cols),
], remainder="drop")

hgb_pipe = Pipeline([
    ("prep", clone(dense_prep)),
    ("model", HistGradientBoostingClassifier(
        learning_rate=0.08, max_iter=250, max_leaf_nodes=31,
        min_samples_leaf=30, early_stopping=True,
        random_state=RANDOM_STATE,
    )),
])
hgb_param_distributions = {
    "model__learning_rate": [0.04, 0.07, 0.10],
    "model__max_iter": [150, 250, 350],
    "model__max_leaf_nodes": [15, 31, 63],
    "model__min_samples_leaf": [20, 50],
    "model__l2_regularization": [0.0, 1.0],
}
hgb_search = RandomizedSearchCV(
    hgb_pipe, param_distributions=hgb_param_distributions,
    n_iter=10, scoring="roc_auc", cv=3, random_state=RANDOM_STATE,
    n_jobs=N_JOBS, return_train_score=True, refit=True, error_score="raise",
)
hgb_search.fit(X_train, y_train)
print("Лучшие параметры HGB:", hgb_search.best_params_)
print(f"Search CV ROC AUC (3 folds): {hgb_search.best_score_:.4f}")

hgb_search_results = pd.DataFrame(hgb_search.cv_results_).sort_values("mean_test_score", ascending=False)
display(hgb_search_results[["param_model__learning_rate", "param_model__max_iter",
                            "param_model__max_leaf_nodes", "param_model__min_samples_leaf",
                            "param_model__l2_regularization", "mean_train_score",
                            "mean_test_score", "std_test_score", "mean_fit_time"]].round(4))

# Валидационная карта показывает средний CV-score среди проверенных комбинаций.
hgb_pivot = hgb_search_results.pivot_table(
    index="param_model__learning_rate", columns="param_model__max_iter",
    values="mean_test_score", aggfunc="max")
plt.figure(figsize=(7, 4.5))
sns.heatmap(hgb_pivot, annot=True, fmt=".3f", cmap="viridis",
            cbar_kws={"label": "Best search CV ROC AUC"})
plt.xlabel("max_iter")
plt.ylabel("learning_rate")
plt.title("HGB: CV ROC AUC по learning_rate × max_iter")
plt.tight_layout()
plt.show()

# Сравниваем выбранный HGB с ансамблями на одном и том же 5-fold split.
hgb_best_pipe = clone(hgb_search.best_estimator_)
hgb_cv_out = cross_validate(hgb_best_pipe, X_train, y_train, cv=cv,
                            scoring={"roc_auc": "roc_auc", "pr_auc": "average_precision",
                                     "bal_acc": "balanced_accuracy", "neg_log_loss": "neg_log_loss"},
                            n_jobs=N_JOBS, return_train_score=True, error_score="raise")
hgb_cv_auc = float(hgb_cv_out["test_roc_auc"].mean())
hgb_cv_auc_std = float(hgb_cv_out["test_roc_auc"].std(ddof=1))
hgb_cv_fit_s = float(hgb_cv_out["fit_time"].mean())
hgb_cv_table = pd.DataFrame([{
    "model": "HistGradientBoosting", "roc_auc_mean": hgb_cv_auc,
    "roc_auc_std": hgb_cv_auc_std, "pr_auc_mean": hgb_cv_out["test_pr_auc"].mean(),
    "balanced_acc_mean": hgb_cv_out["test_bal_acc"].mean(),
    "log_loss_mean": (-hgb_cv_out["test_neg_log_loss"]).mean(),
    "fit_s_mean": hgb_cv_fit_s,
}])
display(hgb_cv_table.round(4))

## 10. Финальное сравнение на test

После завершения CV/OOB-подбора каждая фиксированная конфигурация обучается на всём `X_train`. Затем вычисляются итоговые метрики на `X_test`, время обучения, время предсказания и сериализованный размер модели. Таблица использует порог 0.5 для согласованного сравнения классификационных метрик; специальный порог с ограничением recall будет выбран отдельно, только по out-of-fold прогнозам train.

In [ ]:
# Собираем фиксированные конфигурации. Здесь уже не подбираются параметры по test.
pruned_model_params = {key.replace("model__", ""): value
                        for key, value in best_pruned_params.items()}
final_models = {
    "Dummy": Pipeline([("prep", clone(prep)), ("model", DummyClassifier(strategy="prior"))]),
    "Logistic Regression": Pipeline([("prep", clone(logreg_prep)),
                                      ("model", LogisticRegression(max_iter=1500, solver="lbfgs"))]),
    "Decision Tree": Pipeline([("prep", clone(prep)),
                                ("model", DecisionTreeClassifier(**selected_tree_params,
                                                                  random_state=RANDOM_STATE))]),
    "Pruned Decision Tree": Pipeline([("prep", clone(prep)),
                                       ("model", DecisionTreeClassifier(random_state=RANDOM_STATE,
                                                                         **pruned_model_params))]),
    "Random Forest": Pipeline([("prep", clone(prep)),
                                ("model", RandomForestClassifier(
                                    n_estimators=rf_n_estimators_selected,
                                    max_features=rf_selected_params["max_features"],
                                    min_samples_leaf=rf_selected_params["min_samples_leaf"],
                                    max_samples=rf_selected_params["max_samples"],
                                    bootstrap=True, oob_score=True, n_jobs=1,
                                    random_state=RANDOM_STATE))]),
    "Extra Trees": Pipeline([("prep", clone(prep)),
                             ("model", ExtraTreesClassifier(
                                 n_estimators=rf_n_estimators_selected,
                                 min_samples_leaf=rf_selected_params["min_samples_leaf"],
                                 max_features=rf_selected_params["max_features"],
                                 n_jobs=1, random_state=RANDOM_STATE))]),
    "HistGradientBoosting": clone(hgb_search.best_estimator_),
}

# Общий CV-ROC-AUC для выбора лучшего ансамбля (все три использовали одинаковые 5 фолдов).
cv_auc_by_model = {
    "Dummy": baseline_cv_stats["Dummy"]["roc_auc_mean"],
    "Logistic Regression": baseline_cv_stats["Logistic Regression"]["roc_auc_mean"],
    "Decision Tree": selected_tree_cv_auc,
    "Pruned Decision Tree": best_pruned_cv_auc,
    "Random Forest": rf_cv_auc,
    "Extra Trees": extra_cv_auc,
    "HistGradientBoosting": hgb_cv_auc,
}
ensemble_cv_auc = {name: cv_auc_by_model[name]
                   for name in ["Random Forest", "Extra Trees", "HistGradientBoosting"]}
ensemble_ranking = sorted(ensemble_cv_auc.items(), key=lambda item: item[1], reverse=True)
best_ensemble_name = ensemble_ranking[0][0]
backup_ensemble_name = ensemble_ranking[1][0]
print("Выбор ансамбля только по одинаковым 5-fold CV оценкам:")
display(pd.DataFrame(ensemble_ranking, columns=["ensemble", "cv_roc_auc"]).round(4))
print("Основной кандидат:", best_ensemble_name, "| Резервный:", backup_ensemble_name)

final_rows = []
test_probabilities = {}
fit_seconds = {}
predict_seconds = {}
model_sizes_mb = {}
for name, fitted in final_models.items():
    t0 = perf_counter()
    fitted.fit(X_train, y_train)
    fit_seconds[name] = perf_counter() - t0

    t0 = perf_counter()
    p = fitted.predict_proba(X_test)[:, 1]
    predict_seconds[name] = perf_counter() - t0
    test_probabilities[name] = p
    pred = (p >= 0.5).astype(int)
    model_sizes_mb[name] = len(pickle.dumps(fitted, protocol=pickle.HIGHEST_PROTOCOL)) / (1024 ** 2)

    final_rows.append({
        "model": name,
        "cv_roc_auc": cv_auc_by_model[name],
        "test_roc_auc": roc_auc_score(y_test, p),
        "test_pr_auc": average_precision_score(y_test, p),
        "test_balanced_acc@0.5": balanced_accuracy_score(y_test, pred),
        "test_f1@0.5": f1_score(y_test, pred, zero_division=0),
        "test_log_loss": log_loss(y_test, p),
        "fit_s": fit_seconds[name],
        "predict_ms_per_1000": predict_seconds[name] / len(X_test) * 1e6,
        "model_size_MB": model_sizes_mb[name],
    })

final_results = pd.DataFrame(final_rows).sort_values("cv_roc_auc", ascending=False).reset_index(drop=True)
display(final_results.round(4))

# Качество pruning: CV-score найден во время настройки; test-сравнение показано отдельно и только теперь.
print("Выбранное pruning-дерево:", best_pruned_params)
print("Глубина / число листьев:", best_pruned_tree.get_depth(), "/", best_pruned_tree.get_n_leaves())

## 11. Интерпретация: permutation importance

Важность считается на отложенных данных по снижению ROC AUC при перестановке исходного признака, с 10 повторениями. Поскольку вызывается готовый `Pipeline` с исходными колонками `X_test`, уровни одного категориального признака оцениваются совместно на уровне исходного поля, а не как отдельные OHE-столбцы. Это удобно для интерпретации, но перестановка коррелированных признаков может недооценивать их совместную предсказательную информацию.

In [ ]:
from sklearn.inspection import permutation_importance

best_model = final_models[best_ensemble_name]
perm = permutation_importance(
    best_model, X_test, y_test, scoring="roc_auc", n_repeats=10,
    random_state=RANDOM_STATE, n_jobs=N_JOBS,
)
importance_table = pd.DataFrame({
    "feature": X_test.columns,
    "permutation_mean": perm.importances_mean,
    "permutation_std": perm.importances_std,
}).sort_values("permutation_mean", ascending=False).reset_index(drop=True)
display(importance_table.head(10).round(5))

imp_top = importance_table.head(10).sort_values("permutation_mean")
plt.figure(figsize=(9, 6))
plt.barh(imp_top["feature"], imp_top["permutation_mean"], xerr=imp_top["permutation_std"], capsize=3)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Среднее снижение ROC AUC после перестановки")
plt.ylabel("Исходный признак")
plt.title(f"Permutation importance (10 повторений): {best_ensemble_name}")
plt.tight_layout()
plt.show()

# Сравнение с impurity importance, если выбранный ансамбль её поддерживает.
best_estimator = best_model.named_steps["model"] if "model" in best_model.named_steps else best_model
if hasattr(best_estimator, "feature_importances_"):
    transformed_names = best_model.named_steps["prep"].get_feature_names_out()
    raw_columns = list(X.columns)
    def original_feature_name(transformed_name):
        name = str(transformed_name)
        for prefix in ("num__", "cat__"):
            if name.startswith(prefix):
                name = name[len(prefix):]
                break
        candidates = [c for c in raw_columns if name == c or name.startswith(str(c) + "_")]
        return max(candidates, key=len) if candidates else name
    impurity = pd.DataFrame({
        "transformed_feature": transformed_names,
        "impurity_importance": best_estimator.feature_importances_,
    })
    impurity["feature"] = impurity["transformed_feature"].map(original_feature_name)
    impurity = impurity.groupby("feature", as_index=False)["impurity_importance"].sum()
    comparison_importance = importance_table.merge(impurity, on="feature", how="left")
    comparison_importance["perm_rank"] = comparison_importance["permutation_mean"].rank(ascending=False, method="min").astype(int)
    comparison_importance["impurity_rank"] = comparison_importance["impurity_importance"].rank(ascending=False, method="min")
    comparison_importance["rank_gap"] = (comparison_importance["perm_rank"] - comparison_importance["impurity_rank"]).abs()
    display(comparison_importance.sort_values("rank_gap", ascending=False).head(10).round(5))
    print("Примеры расхождений (большой rank_gap):")
    display(comparison_importance.sort_values("rank_gap", ascending=False)[
        ["feature", "permutation_mean", "permutation_std", "impurity_importance", "perm_rank", "impurity_rank"]
    ].head(2).round(5))
else:
    comparison_importance = None
    print(f"У {best_ensemble_name} нет атрибута feature_importances_; impurity importance для этой реализации недоступна.")

print("Ограничения: перестановка может нарушать структуру коррелированных признаков; impurity importance склонна отдавать предпочтение признакам с большим числом возможных разбиений. Для OHE уровни агрегируются до исходных полей только в сравнении impurity importance.")

## 12. Выбор порога и аудит подгрупп

Порог выбирается по out-of-fold прогнозам на train: среди порогов с recall не ниже 0.75 выбирается тот, у которого максимальная precision. Затем, после фиксации порога, считаются confusion matrix и recall/FPR на test в целом и по `sex` и укрупнённым категориям `race`. Малые подгруппы (менее 100 объектов) отдельно не интерпретируются. Аудит описывает различия ошибок, но не доказывает их причину; критерии fairness зависят от контекста применения.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, precision_recall_curve, precision_score, recall_score

# Используем выбранный по CV ансамбль и строго out-of-fold предсказания только на train.
validation_probabilities = cross_val_predict(
    clone(final_models[best_ensemble_name]), X_train, y_train,
    cv=cv, method="predict_proba", n_jobs=N_JOBS,
)[:, 1]
precision_curve, recall_curve, thresholds = precision_recall_curve(y_train, validation_probabilities)
valid_threshold_idx = np.flatnonzero(recall_curve[:-1] >= 0.75)
if len(valid_threshold_idx):
    local_best = valid_threshold_idx[np.argmax(precision_curve[:-1][valid_threshold_idx])]
    selected_threshold = float(thresholds[local_best])
    validation_precision = float(precision_curve[local_best])
    validation_recall = float(recall_curve[local_best])
else:
    selected_threshold = 0.5
    validation_precision = float(precision_score(y_train, validation_probabilities >= selected_threshold, zero_division=0))
    validation_recall = float(recall_score(y_train, validation_probabilities >= selected_threshold, zero_division=0))

print(f"Выбранный порог: {selected_threshold:.5f}")
print(f"OOF train: precision={validation_precision:.4f}; recall={validation_recall:.4f}")

# Test применяется после фиксации модели и порога.
test_p = test_probabilities[best_ensemble_name]
test_pred = (test_p >= selected_threshold).astype(int)
cm = confusion_matrix(y_test, test_pred, labels=[0, 1])
display(pd.DataFrame(cm, index=["actual <=50K", "actual >50K"],
                     columns=["predicted <=50K", "predicted >50K"]))

def binary_group_metrics(y_true_group, y_pred_group, y_score_group=None):
    yt = np.asarray(y_true_group, dtype=int)
    yp = np.asarray(y_pred_group, dtype=int)
    tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
    return {
        "n": int(len(yt)),
        "positive_rate": float(yt.mean()) if len(yt) else np.nan,
        "predicted_positive_rate": float(yp.mean()) if len(yp) else np.nan,
        "precision": float(tp / (tp + fp)) if (tp + fp) else np.nan,
        "recall": float(tp / (tp + fn)) if (tp + fn) else np.nan,
        "FPR": float(fp / (fp + tn)) if (fp + tn) else np.nan,
    }

print("Общие test-метрики при выбранном пороге:")
display(pd.DataFrame([binary_group_metrics(y_test, test_pred)], index=[best_ensemble_name]).round(4))

audit = X_test[[c for c in ["sex", "race"] if c in X_test.columns]].reset_index(drop=True).copy()
audit["y_true"] = y_test.to_numpy()
audit["y_score"] = test_p
audit["y_pred"] = test_pred
audit["sex_group"] = audit["sex"].fillna("Missing").astype(str).str.strip() if "sex" in audit else "Unavailable"
if "race" in audit:
    race_values = audit["race"].fillna("Missing").astype(str).str.strip()
    audit["race_group"] = np.where(race_values.isin(["White", "Black"]), race_values, "Other / combined")
else:
    audit["race_group"] = "Unavailable"

def make_group_table(frame, group_col, min_n=100):
    rows = []
    for group, part in frame.groupby(group_col, dropna=False):
        metrics = binary_group_metrics(part["y_true"], part["y_pred"])
        rows.append({"group": str(group), **metrics, "reported": metrics["n"] >= min_n})
    result = pd.DataFrame(rows).sort_values("n", ascending=False)
    return result

sex_audit = make_group_table(audit, "sex_group")
race_audit = make_group_table(audit, "race_group")
print("Аудит по sex (reported=False означает n<100; оценки такой группы не следует обобщать):")
display(sex_audit.round(4))
print("Аудит по race: White и Black отдельно, остальные категории объединены:")
display(race_audit.round(4))

plt.figure(figsize=(8, 4))
plot_audit = pd.concat([
    sex_audit.assign(attribute="sex"),
    race_audit.assign(attribute="race"),
], ignore_index=True)
sns.barplot(data=plot_audit, x="group", y="FPR", hue="attribute", errorbar=None)
plt.xticks(rotation=25, ha="right")
plt.ylabel("False positive rate")
plt.xlabel("Группа")
plt.title("FPR по укрупнённым подгруппам (test)")
plt.tight_layout()
plt.show()

print("Ограничения fairness-аудита: тестовая выборка содержит только исторические метки; категории могут иметь разный размер и базовую долю положительного класса. Равенство FPR/recall не гарантирует справедливость по всем определениям, а различия метрик не свидетельствуют о причинном эффекте защищённого признака.")

## 13. Заключение и краткая рекомендация техническому руководителю

Итог ниже формируется на основании результатов этого же ноутбука после выполнения всех предыдущих ячеек. Главный кандидат и резерв выбираются **по среднему 5-fold CV ROC AUC среди трёх ансамблей**, а test используется только для финального измерения. Числа будут подставлены автоматически — отдельный отчёт не нужен.

При внедрении особенно важны: (1) репрезентативность данных и сдвиг распределения; (2) ошибки по социальным подгруппам и риск дискриминационного применения; (3) калибровка вероятностей и выбранный порог; (4) воспроизводимость preprocessing и новых категорий. Рекомендуемые проверки после запуска: мониторинг распределений входов и доли пропусков, контроль качества/калибровки по свежим размеченным данным, регулярный аудит ошибок по подгруппам и отслеживание задержки/ошибок сервиса.

In [ ]:
# Динамический итоговый мини-отчёт в этом же ipynb.
results_by_name = final_results.set_index("model")
best_result = results_by_name.loc[best_ensemble_name]
backup_result = results_by_name.loc[backup_ensemble_name]
dummy_result = results_by_name.loc["Dummy"]
logreg_result = results_by_name.loc["Logistic Regression"]
auc_gain_dummy = float(best_result["test_roc_auc"] - dummy_result["test_roc_auc"])
auc_gain_logreg = float(best_result["test_roc_auc"] - logreg_result["test_roc_auc"])

recommendation = f"""
### Рекомендация

**Основная модель:** {best_ensemble_name}. **Резерв:** {backup_ensemble_name}. Выбор сделан по 5-fold CV ROC AUC, не по test.

Числовые аргументы:

1. Средний 5-fold CV ROC AUC основной модели: **{ensemble_cv_auc[best_ensemble_name]:.4f}**.
2. Итоговый test ROC AUC: **{best_result['test_roc_auc']:.4f}**; разница с Dummy по test ROC AUC: **{auc_gain_dummy:+.4f}**.
3. Итоговый test PR AUC: **{best_result['test_pr_auc']:.4f}**; разница с логистической регрессией по test ROC AUC: **{auc_gain_logreg:+.4f}**.
4. Время полного обучения на train: **{best_result['fit_s']:.2f} с**, задержка предсказания: **{best_result['predict_ms_per_1000']:.2f} мс на 1000 объектов**, сериализованный размер: **{best_result['model_size_MB']:.2f} MiB**.

**Компромисс.** Основная модель рекомендована по дискриминационной способности на CV; резерв — следующая по CV ансамблевая модель. Для требований к прозрачности следует сохранять диагностическое/pruned-дерево как объяснимый ориентир, но не подменять им выбранный ансамбль, если качество существенно ниже.

**Риски:** (1) исторические данные Adult могут быть устаревшими и нерепрезентативными; (2) `sex`/`race` и их прокси способны создавать неодинаковые ошибки; (3) порог recall≥0.75 не гарантирует оптимальную калибровку или справедливость; (4) новые категории, пропуски и drift могут ухудшить качество; (5) оценка относится к конкретному split и не доказывает причинных связей.

**Проверки после запуска:** (1) мониторить drift признаков, долю пропусков и неизвестных категорий; (2) на свежих размеченных данных измерять ROC AUC, PR AUC, log loss/Brier score и калибровку; (3) регулярно сравнивать recall/FPR/precision по подгруппам при достаточном числе наблюдений; (4) отслеживать задержку, ошибки пайплайна и размер модели; (5) пересматривать модель и порог при подтверждённом сдвиге данных.

**Ограничение применения:** доход в историческом наборе — социально чувствительная цель. Эти результаты являются учебным прогнозным экспериментом и не обосновывают автоматические решения о трудоустройстве, кредитовании или доступе к услугам.
"""
display(Markdown(recommendation))

print("Параметры воспроизводимости:", {
    "random_state": RANDOM_STATE,
    "train_test_split": {"test_size": 0.20, "stratify": True},
    "cv": "StratifiedKFold(n_splits=5, shuffle=True, random_state=42)",
    "sklearn_version": sklearn.__version__,
    "python_version": sys.version.split()[0],
})